# NumPy Fundamentals & ndarray Architecture Part-II

> This notebook will help you master core array transformations, shape manipulations, axis-based statistics, and memory architectures in NumPy. It covers in-place vs copied sorts, index sorting, structural reshaping, dimensional concatenation, and array generation functions like eye, arange, and linspace with full code examples.

Topics Covered:

* NumPy contiguous memory model vs. Python list pointer overhead
* 1D, 2D, and 3D array creation and axis semantics (np.zeros, np.ones, np.full, np.eye)
* Array generation routines and spacing (np.arange, np.linspace, np.random.random)
* Axis transformations, transposing methods (arr.transpose(), .T), and memory implications
* Structural reshaping, flattening, and dimension inference via -1 (arr.reshape())
* Dimensional concatenation (np.concatenate) and splitting (np.split) along axes 0 and 1
* Sorting routines: in-place (arr.sort()) vs. copied (np.sort()) sorts, and indirect index sorting (np.argsort()) across multiple axes
* Axis-based statistical reductions and aggregations (np.mean, np.median, np.std, np.min, np.max, np.sum)
* Vector dot products and matrix multiplication (np.dot, np.matmul, @)
* Explain broadcasting mechanics, trailing alignment, and vectorized arithmetic
* Cover array querying and selection (np.where, argmax/argmin, boolean masks)
* Detail numerical utilities (np.clip, np.unique, np.nanmean, nan_to_num)


### 1. How does the arr.transpose() method operate in NumPy, and what are its memory implications?

**Technical Breakdown**:
`arr.transpose()` permutes the axes of an array. For a 2D matrix, it reverses index axes (transforming shape `(M, N)` to `(N, M)`). It returns a view of the original array whenever possible rather than copying data into a new buffer, meaning changes to the transposed view reflect in the original memory unless an explicit `.copy()` is triggered. The original array retains its initial layout and remains completely unmodified. Time complexity is $\mathcal{O}(1)$ as only shape and stride metadata are altered.

In [1]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
transposed = arr.transpose()

print("Original:\n", arr)
print("Transposed:\n", transposed)
print("Is View:", transposed.base is arr)

Original:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
Transposed:
 [[1 4 7]
 [2 5 8]
 [3 6 9]]
Is View: True


### 2. How does the .T attribute differ mechanically from arr.transpose()?

**Technical Breakdown**:
The `.T` attribute is an accessor property that serves as syntactic shorthand for calling `arr.transpose()` without arguments. For a 2D array, it swaps axis 0 and axis 1. Like `arr.transpose()`, `.T` alters the array strides without reallocating or copying underlying memory buffers in CPython. Time complexity is $\mathcal{O}(1)$, and space complexity is $\mathcal{O}(1)$.

In [2]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
t_arr = arr.T
print("Array via .T:\n", t_arr)
print("Shares Memory:", np.shares_memory(arr, t_arr))


Array via .T:
 [[1 4 7]
 [2 5 8]
 [3 6 9]]
Shares Memory: True


### 3. What are the defaults and behavior of np.sort(array, axis=-1)?

**Technical Breakdown**:
`np.sort()` returns a sorted copy of an array without modifying the source object. The parameter `axis` specifies along which dimension to sort; its default value is `-1`, which directs sorting along the last axis. For a 2D array, the last axis corresponds to the columns (`axis=1`), causing each row to be sorted independently in ascending order. Time complexity is $\mathcal{O}(N \log N)$ per slice, governed by Quicksort by default.

In [3]:
import numpy as np

arr = np.array([[5, 2, 3], [8, 1, 6], [7, 4, 9]])
sorted_copy = np.sort(arr, axis=-1)

print("Original Array:\n", arr)
print("np.sort(axis=-1):\n", sorted_copy)

Original Array:
 [[5 2 3]
 [8 1 6]
 [7 4 9]]
np.sort(axis=-1):
 [[2 3 5]
 [1 6 8]
 [4 7 9]]


### 4. How does np.sort(arr, axis=0) reorder multidimensional data?

**Technical Breakdown**:
Passing `axis=0` instructs NumPy to execute operations vertically down columns. In a 2D matrix of shape `(M, N)`, sorting along `axis=0` rearranges the elements of each column independently across the row indices. Values across different columns are not compared or swapped together. Space complexity involves allocating a new array of matching shape `(M, N)`.

In [4]:
import numpy as np

arr = np.array([[5, 2, 3], [8, 1, 6], [7, 4, 9]])
sorted_axis0 = np.sort(arr, axis=0)

print("Sorted along axis 0 (columns):\n", sorted_axis0)

Sorted along axis 0 (columns):
 [[5 1 3]
 [7 2 6]
 [8 4 9]]


### 5. How does np.sort(arr, axis=1) process rows in a 2D array?

**Technical Breakdown**:
Passing `axis=1` directs execution horizontally across columns, operating along each individual row. For a 2D array, this sorts each row independently in ascending order while keeping rows confined to their existing vertical positions.

In [5]:
import numpy as np

arr = np.array([[3, 2, 1], [6, 5, 4]])
sorted_axis1 = np.sort(arr, axis=1)

print("Sorted along axis 1 (rows):\n", sorted_axis1)

Sorted along axis 1 (rows):
 [[1 2 3]
 [4 5 6]]


### 6. How does the in-place method arr.sort() differ from the top-level function np.sort()?

**Technical Breakdown**:
`arr.sort(axis=-1)` mutates the underlying memory buffer of the array directly and returns `None`. Conversely, `np.sort()` allocates and returns an entirely new array containing sorted elements, leaving the source array intact. In-place sorting eliminates buffer allocation overhead, running in $\mathcal{O}(1)$ auxiliary space.

In [6]:
import numpy as np

arr = np.array([[5, 2, 3], [8, 1, 6], [7, 4, 9]])
result = arr.sort(axis=-1)

print("Return value:", result)
print("Mutated original array:\n", arr)

Return value: None
Mutated original array:
 [[2 3 5]
 [1 6 8]
 [4 7 9]]


### 7. What is the operational purpose and return type of np.argsort() on a 1D array?

**Technical Breakdown**:
`np.argsort()` performs an indirect sort, returning an `ndarray` of integers (`int64` or `int32`) representing the indices that partition or sort the array in ascending order. The original array is not modified. These indices can subsequently be passed via fancy indexing to reconstruct the sorted sequence or align parallel arrays.

In [7]:
import numpy as np

arr1 = np.array([3, 1, 5, 2, 4])
indices = np.argsort(arr1)

print("Original Array:", arr1)
print("Sorted Indices:", indices)
print("Reconstructed via Fancy Indexing:", arr1[indices])

Original Array: [3 1 5 2 4]
Sorted Indices: [1 3 0 4 2]
Reconstructed via Fancy Indexing: [1 2 3 4 5]


### 8. How does np.argsort(arr, axis=1) evaluate multidimensional rows?

**Technical Breakdown**:
When applied to a 2D array with `axis=1`, `np.argsort()` returns a 2D integer index array with the same shape. Each row in the resulting array contains index positions corresponding to the sorted order of values across that specific row.

In [8]:
import numpy as np

arr2 = np.array([[3, 2, 1], [6, 4, 5]])
row_indices = np.argsort(arr2, axis=1)

print("Row-wise argsort indices:\n", row_indices)

Row-wise argsort indices:
 [[2 1 0]
 [1 2 0]]


### 9. How does np.argsort(arr, axis=0) calculate indices across column dimensions?

**Technical Breakdown**:
With `axis=0`, sorting indices are determined down each column independently. The resulting index at position `(r, c)` indicates which original row index contains the $r$-th smallest element of column $c$.

In [9]:
import numpy as np

arr3 = np.array([[3, 5, 1], [6, 2, 4]])
col_indices = np.argsort(arr3, axis=0)

print("Column-wise argsort indices:\n", col_indices)

Column-wise argsort indices:
 [[0 1 0]
 [1 0 1]]


### 10. How does np.concatenate join multiple 1D arrays?

**Technical Breakdown**:
`np.concatenate((a1, a2, ...), axis=0)` joins a sequence of arrays along an existing axis. For 1D arrays, the default `axis=0` merges arrays end-to-end into a single contiguous 1D array whose length equals the sum of the input lengths. This creates a new memory buffer of size $\sum N_i$.

In [10]:
import numpy as np

arr1 = np.array([1, 2, 3])
arr2 = np.array([4, 5, 6])
result = np.concatenate((arr1, arr2),axis=0)

print("Concatenated 1D Array:", result)
print("Shape:", result.shape)

Concatenated 1D Array: [1 2 3 4 5 6]
Shape: (6,)


### 11. What structural constraints govern np.concatenate along axis=0 for 2D arrays?

**Technical Breakdown**:
Concatenating 2D arrays along `axis=0` stacks them vertically by rows. This requires all input arrays to have identical shapes across all dimensions except the concatenation dimension (`axis=0`); specifically, they must possess the exact same number of columns.

In [11]:
import numpy as np

arr1 = np.array([[1, 2, 3], [4, 5, 6]])
arr3 = np.array([[11, 12, 13]])
result = np.concatenate((arr1, arr3), axis=0)

print("Concatenated along axis 0:\n", result)
print("Result Shape:", result.shape)

Concatenated along axis 0:
 [[ 1  2  3]
 [ 4  5  6]
 [11 12 13]]
Result Shape: (3, 3)


### 12. What structural constraints govern np.concatenate along axis=1 for 2D arrays?

**Technical Breakdown**:
When combining arrays along `axis=1`, arrays are joined horizontally across columns. All candidate arrays must match in size across dimension 0 (they must have the exact same number of rows).

In [12]:
import numpy as np

arr1 = np.array([[1, 2, 3], [4, 5, 6]])
arr2 = np.array([[7, 8], [9, 10]])
result = np.concatenate((arr1, arr2), axis=1)

print("Concatenated along axis 1:\n", result)
print("Result Shape:", result.shape)

Concatenated along axis 1:
 [[ 1  2  3  7  8]
 [ 4  5  6  9 10]]
Result Shape: (2, 5)


### 13. What exception is raised when np.concatenate encounters dimensional incompatibility?

**Technical Breakdown**:
If inputs have mismatched dimensions along any axis other than the specified concatenation axis, NumPy raises a `ValueError`. C-level buffer allocation fails because a non-ragged dense `ndarray` must maintain rectangular symmetry across all coordinate axes.

In [13]:
import numpy as np

arr1 = np.array([[1, 2, 3], [4, 5, 6]])  # shape: (2, 3)
arr2 = np.array([[7, 8], [9, 10]])        # shape: (2, 2)

try:
    np.concatenate((arr1, arr2), axis=0)
except ValueError as exc:
    print(f"Caught expected exception: {type(exc).__name__} -> {exc}")

Caught expected exception: ValueError -> all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 3 and the array at index 1 has size 2


### 14. How does arr.reshape() construct new view shapes from flat or existing arrays?

**Technical Breakdown**:
`arr.reshape(*shape)` returns an array with a modified dimension tuple without altering the underlying data. By default, elements are read and populated in C-contiguous row-major order. The product of dimensions in the requested shape must equal the array's total element count (`arr.size`).

In [14]:
import numpy as np

arr1d = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
arr2d = arr1d.reshape(4, 3)
arr3d = arr1d.reshape(2, 3, 2)

print("Reshaped (4, 3):\n", arr2d)
print("Reshaped (2, 3, 2):\n", arr3d)

Reshaped (4, 3):
 [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
Reshaped (2, 3, 2):
 [[[ 0  1]
  [ 2  3]
  [ 4  5]]

 [[ 6  7]
  [ 8  9]
  [10 11]]]


### 15. What is the semantic role of -1 in arr.reshape()?

**Technical Breakdown**:
Passing `-1` for a dimension in `arr.reshape()` delegates calculation to NumPy, which infers the correct dimension size by computing $N = \frac{\text{arr.size}}{\prod(\text{specified dimensions})}$. Only a single dimension may be designated as `-1`. When used alone as `.reshape(-1)`, it collapses the array into a flattened 1D array.

In [15]:
import numpy as np

arr2d = np.array([[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]])
flattened = arr2d.reshape(-1)
auto_rows = arr2d.reshape(-1, 2)

print("Flattened via -1:", flattened)
print("Inferred Rows (Shape):\n", auto_rows.shape)

Flattened via -1: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Inferred Rows (Shape):
 (6, 2)


### 16. Why does arr.reshape() raise a ValueError on incompatible shapes?

**Technical Breakdown**:
Reshaping requires the total number of elements to remain invariant. If an array of size $N$ is requested to reshape into dimensions whose product does not equal $N$, a `ValueError` is raised because NumPy arrays cannot contain empty or uninitialized memory slots.

In [16]:
import numpy as np

arr = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])  # 10 elements

try:
    arr.reshape(3, 4)  # Requires 12 elements
except ValueError as exc:
    print(f"Caught expected exception: {type(exc).__name__} -> {exc}")

Caught expected exception: ValueError -> cannot reshape array of size 10 into shape (3,4)


### 17. How does np.split divide a 1D array into equal partitions?

**Technical Breakdown**:
`np.split(ary, indices_or_sections, axis=0)` breaks an array into multiple sub-arrays. When passed an integer $N$ for `indices_or_sections`, it splits the array along the specified axis into $N$ equal chunks. It returns a Python list of `ndarray` objects. The target dimension length must be divisible by $N$, otherwise a `ValueError` is raised.

* Syntax: np.split(array, indices_or_sections, axis=0)
  - array: Array to split.
  - indices_or_sections: Number of equal sections or indices where to split.
  - axis: Axis along which to split. Default is 0.
  ```python
  import numpy as np

  arr = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
  result = np.split(arr, [3, 6])
  print(result)
  ```
  Output:
  ```python
  [array([10, 20, 30]), array([40, 50, 60]), array([ 70,  80,  90, 100])]
  ```

In [17]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
sub_arrays = np.split(arr, 5)

print("Split count:", len(sub_arrays))
print("Parts:", [p.tolist() for p in sub_arrays])

Split count: 5
Parts: [[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]]


### 18. How does np.split segment a 2D array along axis=1?

**Technical Breakdown**:
Splitting along `axis=1` partitions columns horizontally. For a matrix with shape `(M, N)`, dividing into $K$ sections splits the $N$ columns into $K$ separate sub-arrays, each retaining all $M$ rows and having shape `(M, N/K)`.


In [18]:
import numpy as np

arr_2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
col_splits = np.split(arr_2d, 3, axis=1)

for idx, part in enumerate(col_splits):
    print(f"Split {idx} (axis=1):\n{part}")

Split 0 (axis=1):
[[1]
 [4]
 [7]]
Split 1 (axis=1):
[[2]
 [5]
 [8]]
Split 2 (axis=1):
[[3]
 [6]
 [9]]


### 19. How does np.split segment a 2D array along axis=0?

**Technical Breakdown**:
Splitting along `axis=0` partitions rows vertically. For an `(M, N)` array, dividing into $K$ sections produces $K$ sub-arrays of shape `(M/K, N)`.

In [19]:
import numpy as np

arr_2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
row_splits = np.split(arr_2d, 3, axis=0)

for idx, part in enumerate(row_splits):
    print(f"Split {idx} (axis=0):\n{part}")

Split 0 (axis=0):
[[1 2 3]]
Split 1 (axis=0):
[[4 5 6]]
Split 2 (axis=0):
[[7 8 9]]


### 20. How does np.mean compute arithmetic averages across global, row, and column axes?

**Technical Breakdown**:
`np.mean(a, axis=None)` calculates the arithmetic mean. If `axis=None`, it computes the mean of all elements flattened into a single float scalar. When `axis=0`, it averages across rows down each column, returning shape `(N,)`. When `axis=1`, it averages across columns along each row, returning shape `(M,)`.

In [20]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

print("Global Mean:", np.mean(arr))
print("Axis 0 Mean (Columns):", np.mean(arr, axis=0)) # np.mean(arr, axis=0, dtype=int/float)
print("Axis 1 Mean (Rows):", np.mean(arr, axis=1))

Global Mean: 3.5
Axis 0 Mean (Columns): [2.5 3.5 4.5]
Axis 1 Mean (Rows): [2. 5.]


### 21. How does np.median evaluate positional medians across different axes?

**Technical Breakdown**:
`np.median(a, axis=None)` sorts elements along the target axis and returns the 50th percentile value. For an even count of elements $N$, it calculates the mean of the two central sorted values. Unlike `np.mean`, it cannot be computed as a running sum and requires partial sorting or partitioning ($\mathcal{O}(N)$ using `introselect`).

In [21]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

print("Median Global:", np.median(arr))
print("Median Axis 0:", np.median(arr, axis=0))
print("Median Axis 1:", np.median(arr, axis=1))

Median Global: 3.5
Median Axis 0: [2.5 3.5 4.5]
Median Axis 1: [2. 5.]


### 22. What algorithm and reduction does np.std execute across axes?

**Technical Breakdown**:
`np.std(a, axis=None)` computes the population standard deviation: $\sigma = \sqrt{\frac{1}{N}\sum (x_i - \bar{x})^2}$ with degrees of freedom `ddof=0`. It returns a floating-point scalar if `axis=None`, or an array corresponding to the reduced dimensions when `axis=0` or `axis=1`.

In [22]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

print("Std Global:", np.std(arr))
print("Std Axis 0:", np.std(arr, axis=0))
print("Std Axis 1:", np.std(arr, axis=1))

Std Global: 1.707825127659933
Std Axis 0: [1.5 1.5 1.5]
Std Axis 1: [0.81649658 0.81649658]


### 23. How do np.min and np.max operate on unreduced global arrays?

**Technical Breakdown**:
When called with `axis=None` (the default), `np.min` and `np.max` search the entire contiguous memory block to identify the absolute minimum and maximum values, returning a single scalar. Time complexity is $\mathcal{O}(N)$ where $N = \text{arr.size}$.

In [23]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

print("Global Min:", np.min(arr))
print("Global Max:", np.max(arr))

Global Min: 1
Global Max: 6


### 24. How do np.min and np.max reduce 2D structures along axis=0 and axis=1?

**Technical Breakdown**:
Providing an explicit `axis` parameter collapses that specific dimension by taking the minimum or maximum across those indices. For shape `(M, N)`, `axis=0` reduces across the $M$ rows producing shape `(N,)`, while `axis=1` reduces across the $N$ columns producing shape `(M,)`.

In [24]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

print("Min Axis 0 (Columns):", np.min(arr, axis=0))
print("Min Axis 1 (Rows):", np.min(arr, axis=1))
print("Max Axis 0 (Columns):", np.max(arr, axis=0))
print("Max Axis 1 (Rows):", np.max(arr, axis=1))

Min Axis 0 (Columns): [1 2 3]
Min Axis 1 (Rows): [1 4]
Max Axis 0 (Columns): [4 5 6]
Max Axis 1 (Rows): [3 6]


### 25. How does np.sum execute reduction operations along axes?

**Technical Breakdown**:
`np.sum(a, axis=None)` aggregates array elements using vector registers (SIMD). When `axis=None`, it sums all elements into a scalar. When `axis=0`, it sums vertically down columns, while `axis=1` sums horizontally along rows.

In [25]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

print("Sum Global:", np.sum(arr))
print("Sum Axis 0:", np.sum(arr, axis=0))
print("Sum Axis 1:", np.sum(arr, axis=1))

Sum Global: 21
Sum Axis 0: [5 7 9]
Sum Axis 1: [ 6 15]


### 26. What are the key architectural differences between a Python list and a numpy.ndarray?

**Technical Breakdown**:
A Python `list` is a dynamic array storing pointers (`PyObject*`) to heterogeneous objects scattered across heap memory. Arithmetic operations cannot execute element-wise directly without explicit loops. A `numpy.ndarray` stores homogeneous data in a contiguous C-level memory buffer. This enables direct vectorization, cache locality, and SIMD instruction execution.

In [26]:
import numpy as np

py_list = [1, 2, 3]
np_arr = np.array([1, 2, 3])

# Vectorized operation works on ndarray
print("Array Division:", np_arr / 3)

# Python list cannot divide directly
try:
    _ = py_list / 3
except TypeError as e:
  print(f"List Error: {type(e).__name__} -> {e}")

Array Division: [0.33333333 0.66666667 1.        ]
List Error: TypeError -> unsupported operand type(s) for /: 'list' and 'int'


### 27. What are the parameter rules and default dtype for np.zeros()?

**Technical Breakdown**:
`np.zeros(shape, dtype=float)` allocates a block of memory filled with zeros matching the given `shape` (an `int` for 1D or a `tuple` of ints for multidimensional arrays). The default `dtype` is `numpy.float64`.

In [27]:
import numpy as np

z_1d = np.zeros(5)
z_2d = np.zeros((3, 4))

print("1D zeros dtype:", z_1d.dtype)
print("2D zeros:\n", z_2d)

1D zeros dtype: float64
2D zeros:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


### 28. How does np.zeros() handle an explicit integer dtype?

**Technical Breakdown**:
Setting `dtype=int` forces NumPy to cast memory representations to fixed-width machine integers (`int64` or `int32`), omitting floating-point representations (`0` instead of `0.`).

In [28]:
import numpy as np

z_int = np.zeros((2, 3), dtype=int)

print("Integer Zeros:\n", z_int)
print("Dtype:", z_int.dtype)

Integer Zeros:
 [[0 0 0]
 [0 0 0]]
Dtype: int64


### 29. What is the behavior of np.ones() with default parameters?

**Technical Breakdown**:
`np.ones(shape, dtype=float)` allocates an array filled with values of `1.0`. Memory is configured with the given shape and default float dtype (`np.float64`).

In [29]:
import numpy as np

ones_arr = np.ones(3)

print("1D Ones:", ones_arr)
print("Dtype:", ones_arr.dtype)

1D Ones: [1. 1. 1.]
Dtype: float64


### 30. How does np.ones() behave when dtype=int is explicitly provided?

**Technical Breakdown**:
Specifying `dtype=int` allocates machine integers initialized with `1`. Each entry uses 8 bytes (on 64-bit platforms) without decimal values.

In [30]:
import numpy as np

ones_int = np.ones((2, 3), dtype=int)

print("Integer Ones:\n", ones_int)
print("Dtype:", ones_int.dtype)

Integer Ones:
 [[1 1 1]
 [1 1 1]]
Dtype: int64


### 31. How does np.full() determine its output array dtype?

**Technical Breakdown**:
`np.full(shape, fill_value)` allocates an array of the requested shape filled with `fill_value`. If no explicit `dtype` is provided, NumPy infers the dtype directly from the scalar type of `fill_value` (e.g., Python `int` infers `int64`, while `float` infers `float64`).

In [31]:
import numpy as np

f_int = np.full((2, 2),-1)
f_float = np.full((2, 2), 3.14)

print("Int full dtype:", f_int.dtype, "\n", f_int)
print("Float full dtype:", f_float.dtype, "\n", f_float)

Int full dtype: int64 
 [[-1 -1]
 [-1 -1]]
Float full dtype: float64 
 [[3.14 3.14]
 [3.14 3.14]]


### 32. What are the dimensional constraints and output structure of np.eye(N)?

**Technical Breakdown**:
`np.eye(N)` creates a 2D identity matrix of shape `(N, N)` with ones on the main diagonal and zeros elsewhere. The default `dtype` is `float64`. `np.eye` exclusively outputs 2-dimensional arrays; it cannot directly generate 1D or 3D arrays.

In [32]:
import numpy as np

ident = np.eye(3)

print("Identity Matrix (3x3):\n", ident)
print("Dimensions:", ident.ndim)

Identity Matrix (3x3):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Dimensions: 2


### 33. How does np.eye construct non-square rectangular matrices using (N, M)?

**Technical Breakdown**:
Providing both `N` (number of rows) and `M` (number of columns) generates a 2D matrix of shape `(N, M)`. The main diagonal starts at index `(0, 0)` and sets values to `1.0` wherever row index equals column index up to $\min(N, M)$.

In [33]:
import numpy as np

rect_eye = np.eye(3,4)

print("3x4 Identity Matrix:\n", rect_eye)
print("Shape:", rect_eye.shape)

3x4 Identity Matrix:
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
Shape: (3, 4)


### 34. What is the effect of passing a positive offset k > 0 to np.eye()?

**Technical Breakdown**:
The parameter `k` designates the diagonal index. When $k > 0$, the diagonal containing ones is shifted upward above the main diagonal by $k$ positions. Element $(i, j)$ is set to 1 if and only if $j - i = k$.

In [34]:
import numpy as np

shifted_up = np.eye(4, k=1)

print("Diagonal Shifted (k=1):\n", shifted_up)

Diagonal Shifted (k=1):
 [[0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 0.]]


### 35. What is the effect of passing a negative offset k < 0 to np.eye()?

**Technical Breakdown**:
When $k < 0$, the diagonal containing ones is shifted downward below the main diagonal by $|k|$ positions. Element $(i, j)$ is set to 1 if and only if $i - j = |k|$.

In [35]:
import numpy as np

shifted_down = np.eye(5, k=-2)

print("Diagonal Shifted (k=-2):\n", shifted_down)

Diagonal Shifted (k=-2):
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]]


### 36. How does np.arange() behave when invoked with a single argument?

**Technical Breakdown**:
When passed a single positional argument, `np.arange(stop)` treats that value as an exclusive upper bound. The start value defaults to `0` and step defaults to `1`. It produces a 1D `ndarray`.

In [36]:
import numpy as np

arr = np.arange(10)

print("Range Array:", arr)
print("Length:", len(arr))

Range Array: [0 1 2 3 4 5 6 7 8 9]
Length: 10


### 37. How does np.arange(start, stop) define its interval bounds?

**Technical Breakdown**:
Calling `np.arange(start, stop)` creates an interval that includes `start` and excludes `stop`: $[\text{start}, \text{stop})$. The default step remains `1`.

In [37]:
import numpy as np

arr = np.arange(3, 13)

print("Bound interval [3, 13):", arr)

Bound interval [3, 13): [ 3  4  5  6  7  8  9 10 11 12]


### 38. How does np.arange(start, stop, step) handle striding?

**Technical Breakdown**:
The `step` parameter specifies the spacing between successive elements. Elements are generated according to $x_i = \text{start} + i \cdot \text{step}$ until reaching or exceeding `stop`. When using floating-point steps, floating-point roundoff can make the number of output elements unpredictable.

In [38]:
import numpy as np

arr = np.arange(0, 11, 2)

print("Strided arange (step=2):", arr)

Strided arange (step=2): [ 0  2  4  6  8 10]


### 39. What distributions and bounds govern np.random.random()?

**Technical Breakdown**:
`np.random.random(size)` draws random floating-point values from a continuous uniform distribution over the semi-open interval $[0.0, 1.0)$. The `size` parameter accepts an integer or tuple of integers defining the output array dimensions.

In [39]:
import numpy as np

r_1d = np.random.random(5)
r_2d = np.random.random((2, 3))
r_3d = np.random.random((2, 3, 4))

print("1D Uniform Random:\n", r_1d)
print("2D Uniform Random:\n", r_2d)
print("3D Uniform Random:\n", r_3d)

1D Uniform Random:
 [0.41775435 0.48540634 0.90243454 0.51132509 0.61368433]
2D Uniform Random:
 [[0.10272876 0.58971057 0.00214065]
 [0.00657591 0.93902428 0.37590597]]
3D Uniform Random:
 [[[0.73076535 0.19132845 0.50655254 0.49803282]
  [0.24816635 0.79569985 0.9774758  0.97141136]
  [0.95511348 0.30439899 0.98338633 0.47303485]]

 [[0.40357095 0.70229707 0.93288043 0.50356928]
  [0.08850207 0.10620773 0.45599737 0.37391156]
  [0.965187   0.94121807 0.72870589 0.40010371]]]


### 40. How does np.linspace(start, stop, num) calculate evenly spaced points?

**Technical Breakdown**:
`np.linspace(start, stop, num=50, endpoint=True)` divides the interval $[\text{start}, \text{stop}]$ into `num` evenly spaced points. Unlike `arange`, `np.linspace` includes `stop` by default (`endpoint=True`), using a step size of $\Delta = \frac{\text{stop} - \text{start}}{\text{num} - 1}$.

In [40]:
import numpy as np

points = np.linspace(0, 100, 5)

print("Linspace (5 points [0, 100]):", points)

Linspace (5 points [0, 100]): [  0.  25.  50.  75. 100.]


### 41. How do endpoint=False and retstep=True modify np.linspace?

**Technical Breakdown**:
Setting `endpoint=False` excludes `stop`, creating step sizes of $\Delta = \frac{\text{stop} - \text{start}}{\text{num}}$. Setting `retstep=True` causes `np.linspace` to return a 2-tuple `(samples_array, step_value)` instead of just the array.

In [41]:
import numpy as np

samples, step = np.linspace(0, 20, 10, endpoint=False, retstep=True) #(0, 20, 10, endpoint=False, retstep=True)

print("Samples (endpoint=False):\n", samples)
print("Calculated Step:", step)

Samples (endpoint=False):
 [ 0.  2.  4.  6.  8. 10. 12. 14. 16. 18.]
Calculated Step: 2.0


### 42. How does `np.dot()` compute the inner product of two 1D vectors?

**Technical Breakdown**: When given two 1D arrays of matching length $N$, `np.dot(u, v)` computes the algebraic inner dot product: $\sum_{i=0}^{N-1} u_i \cdot v_i$. It reduces both 1D vectors to a single scalar float or integer in $\mathcal{O}(N)$ time and $\mathcal{O}(1)$ auxiliary space via BLAS level-1 routines (`sdot`/`ddot`). If the vector lengths do not match, NumPy raises a `ValueError`.

In [42]:
import numpy as np

u = np.array([1.5, 2.0, 3.5])
v = np.array([4.0, -1.0, 2.0])

dot_product = np.dot(u, v)
print("Vector u:", u)
print("Vector v:", v)
print("Inner Product:", dot_product)

# Length mismatch check
try:
    np.dot(u, np.array([1.0, 2.0]))
except ValueError as exc:
    print(f"Dimension Mismatch Handled: {type(exc).__name__} -> {exc}")

Vector u: [1.5 2.  3.5]
Vector v: [ 4. -1.  2.]
Inner Product: 11.0
Dimension Mismatch Handled: ValueError -> shapes (3,) and (2,) not aligned: 3 (dim 0) != 2 (dim 0)


### 43. How does `np.dot()` execute 2D matrix multiplication?

**Technical Breakdown**: For two 2D arrays $A$ of shape $(M, K)$ and $B$ of shape $(K, N)$, `np.dot(A, B)` performs standard linear algebraic matrix multiplication, contracting the inner shared dimension $K$ to produce an output matrix of shape $(M, N)$. Computational complexity is $\mathcal{O}(M \cdot K \cdot N)$, executed via BLAS level-3 (`gemm`). A `ValueError` is raised if $A$'s trailing column dimension does not equal $B$'s leading row dimension.

In [43]:
import numpy as np

A = np.array([[1, 2, 3],
              [4, 5, 6]])  # Shape: (2, 3)
B = np.array([[7, 8],
              [9, 1],
              [2, 3]])     # Shape: (3, 2)

C = np.dot(A, B)
print("Matrix A shape:", A.shape)
print("Matrix B shape:", B.shape)
print("Result C (2x2):\n", C)

Matrix A shape: (2, 3)
Matrix B shape: (3, 2)
Result C (2x2):
 [[31 19]
 [85 55]]


### 44. How does `np.matmul` (and the `@` operator) differ mechanically from `np.dot()`?

**Technical Breakdown**: Python 3.5+ introduced the `@` binary infix operator, which invokes `np.matmul()`. While `np.dot` treats 2D matrix multiplication similarly to `np.matmul`, their behavior diverges on higher-dimensional arrays (tensors where `ndim > 2`). `np.dot` calculates sums over the last axis of the first array and the second-to-last axis of the second array, whereas `np.matmul` treats arrays with `ndim > 2` as stacks of 2D matrices broadcast across leading batch dimensions (essential for batch training in deep learning). Furthermore, `np.matmul` prohibits multiplication by scalar primitives.

In [44]:
import numpy as np

# Batched 3D Tensor Multiplication: (Batch_Size, Rows, Cols)
batch_A = np.random.randn(8, 4, 3)
batch_B = np.random.randn(8, 3, 5)

batch_result = batch_A @ batch_B  # Invokes np.matmul
print("Batch Tensor A Shape:", batch_A.shape)
print("Batch Tensor B Shape:", batch_B.shape)
print("Output Batched Shape via @:", batch_result.shape)

# Scalar prohibition
try:
    _ = batch_A @ 2.0
except ValueError as exc:
    print(f"Scalar Matmul Error Handled: {type(exc).__name__} -> {exc}")

Batch Tensor A Shape: (8, 4, 3)
Batch Tensor B Shape: (8, 3, 5)
Output Batched Shape via @: (8, 4, 5)
Scalar Matmul Error Handled: ValueError -> matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)


### 45. What are the operational semantics of element-wise addition versus broadcasting addition?

**Technical Breakdown**: Element-wise addition using `+` or `np.add(A, B)` sums corresponding entries in arrays sharing identical shapes. If shapes differ, NumPy applies its universal broadcasting rules: dimensions are compared starting from trailing (rightmost) dimensions. Two dimensions are compatible if they are equal or if one of them is 1. If an axis has a dimension of 1, it is virtually expanded without replicating data in physical memory, operating in $\mathcal{O}(N)$ total output volume time.

In [45]:
import numpy as np

matrix = np.array([[10, 20, 30],
                   [40, 50, 60]])  # Shape: (2, 3)
row_bias = np.array([1, 2, 3])     # Shape: (3,) -> Broadcasts to (2, 3)
col_bias = np.array([[100],
                     [200]])       # Shape: (2, 1) -> Broadcasts to (2, 3)

print("Add Row Bias:\n", matrix + row_bias)
print("Add Column Bias:\n", matrix + col_bias)

Add Row Bias:
 [[11 22 33]
 [41 52 63]]
Add Column Bias:
 [[110 120 130]
 [240 250 260]]


### 46. How does vectorized boolean masking operate in production feature filtering?

**Technical Breakdown**: Vectorized boolean indexing generates a boolean mask array evaluating element-wise predicates without Python loops. When applied back via `arr[mask]`, NumPy uses non-zero index offsets to extract qualifying items into a new 1D array buffer. Logical operations between arrays require bitwise operators (`&` for AND, `|` for OR, `~` for NOT) because Python's keyword operators `and`/`or` evaluate the truth value of the array container itself, triggering a `ValueError`.

In [46]:
import numpy as np

data = np.array([[22, 105000],
                 [35, 45000],
                 [29, 82000],
                 [42, 120000]])

# Query: Age (col 0) > 25 AND Salary (col 1) > 80,000
age_condition = data[:, 0] > 25
salary_condition = data[:, 1] > 80000
compound_mask = age_condition & salary_condition

filtered_records = data[compound_mask]
print("Boolean Mask:", compound_mask)
print("Filtered Matrix:\n", filtered_records)

Boolean Mask: [False False  True  True]
Filtered Matrix:
 [[    29  82000]
 [    42 120000]]


### 47. How does `np.where()` implement branchless vectorized conditional evaluation?

**Technical Breakdown**: `np.where(condition, [x, y])` functions as a vectorized ternary operator (`x if condition else y`). If all three arguments are supplied, NumPy evaluates `condition` element-wise and yields values selected from `x` or `y`, supporting broadcasting across all three inputs. If only `condition` is passed, it functions as an alias to `np.nonzero(condition)`, returning a tuple of index arrays where the condition holds `True`.

In [47]:
import numpy as np

scores = np.array([45, 88, 72, 33, 95])

# Ternary transformation: Pass/Fail binary flag
status = np.where(scores >= 50, "PASS", "FAIL")
print("Scores:", scores)
print("Status Labels:", status)

# Coordinate extraction variation (only condition supplied)
passing_indices = np.where(scores >= 75)
print("Indices scoring >= 75:", passing_indices[0])


Scores: [45 88 72 33 95]
Status Labels: ['FAIL' 'PASS' 'PASS' 'FAIL' 'PASS']
Indices scoring >= 75: [1 4]


### 48: What is `np.append()` in NumPy? Explain its syntax, parameters, examples, axis behavior, return value, and performance considerations.

**Technical Breakdown**: `np.append()` adds elements to an array and returns a new array without modifying the original array.

* **Syntax:**
    ```python
    np.append(arr, values, axis=None)
    ```

  - `arr`: Original array.
  - `values`: Element(s) to append.
  - `axis`: Axis along which to append. Default is `None`, which flattens the arrays before appending.
* **Key Points:**
  * `axis=None`: Flattens the input arrays before appending.
  * `axis=0`: Adds along the first axis, such as adding a row to a 2D array.
  * `axis=1`: Adds along the second axis, such as adding a column to a 2D array.
  * When an axis is specified, dimensions must match along all other axes.
  * `np.append()` returns a new array; it does not modify the original array in place.
  * Repeated appending can be inefficient because each call creates a new array. For many additions, collect values in a list and convert to a NumPy array afterward.





In [48]:
import numpy as np

# 1D: Append a single element or multiple elements
arr = np.array([10, 20, 30])
print("Original 1D array:", arr)
print("After appending 40:", np.append(arr, 40))
print("After appending 40 and 50:", np.append(arr, [40, 50]))

# 2D: Append a row (axis=0)
arr = np.array([[1, 2], [3, 4]])
print("\nOriginal 2D array:\n", arr)
print("After appending a row along axis=0:\n", np.append(arr, [[5, 6]], axis=0))

# 2D: Append a column (axis=1)
print("After appending a column along axis=1:\n", np.append(arr, [[5], [6]], axis=1))


Original 1D array: [10 20 30]
After appending 40: [10 20 30 40]
After appending 40 and 50: [10 20 30 40 50]

Original 2D array:
 [[1 2]
 [3 4]]
After appending a row along axis=0:
 [[1 2]
 [3 4]
 [5 6]]
After appending a column along axis=1:
 [[1 2 5]
 [3 4 6]]


### 49. How do `np.argmax()` and `np.argmin()` compute extremum indices across axes?

**Technical Breakdown**: `np.argmax(a, axis=None)` and `np.argmin(a, axis=None)` scan array elements to return the indices of maximum or minimum values. In machine learning classification heads, `np.argmax(logits, axis=1)` converts predicted probability matrices of shape `(Batch_Size, Num_Classes)` into 1D class label predictions. When ties occur, the index corresponding to the first occurrence in row-major order is returned.


In [49]:
import numpy as np

# Predicted probabilities for 3 samples across 4 classes
softmax_probs = np.array([[0.1, 0.7, 0.15, 0.05],
                          [0.8, 0.05, 0.1, 0.05],
                          [0.2, 0.3, 0.45, 0.05]])
print("Probability Tensor:\n", softmax_probs)

predicted_classes = np.argmax(softmax_probs, axis=1)
print("Predicted Class Indices per Sample (max, axis=1):", predicted_classes)

predicted_classes = np.argmin(softmax_probs, axis=1)
print("Predicted Class Indices per Sample (min, axis=1):", predicted_classes)

predicted_classes = np.argmax(softmax_probs, axis=0)
print("Predicted Class Indices per Sample (max, axis=0):", predicted_classes)

predicted_classes = np.argmin(softmax_probs, axis=0)
print("Predicted Class Indices per Sample (min, axis=0):", predicted_classes)

Probability Tensor:
 [[0.1  0.7  0.15 0.05]
 [0.8  0.05 0.1  0.05]
 [0.2  0.3  0.45 0.05]]
Predicted Class Indices per Sample (max, axis=1): [1 0 2]
Predicted Class Indices per Sample (min, axis=1): [3 1 3]
Predicted Class Indices per Sample (max, axis=0): [1 0 2 0]
Predicted Class Indices per Sample (min, axis=0): [0 1 1 0]


### 50. What are NumPy's broadcasting rules when combining tensors of unequal dimensions?

**Technical Breakdown**: Broadcasting allows arithmetic operations between arrays of different shapes without copying data in memory. The broadcasting algorithm compares dimensions pairwise from right to left (trailing dimensions first). Two dimensions are compatible if:

1. They are equal, or
2. One of the dimensions is 1.
If an array has fewer dimensions, its shape is prepended with 1s on the left until shapes match in length. If dimensions cannot be aligned under these rules, a `ValueError: operands could not be broadcast together` is raised.

In [50]:
import numpy as np

A = np.ones((4, 3, 1))
B = np.ones((3, 5))

# Comparison:
# A: 4 x 3 x 1
# B: 1 x 3 x 5 (prepends 1 to match 3D)
# Output: 4 x 3 x 5
result = A + B
print("Shape A:", A.shape)
print("Shape B:", B.shape)
print("Broadcast Result Shape:", result.shape)

# Incompatible shape verification
try:
    _ = np.ones((3, 3)) + np.ones((3, 1))
    print("Broadcasting Incompatibility Verified", _.shape)
except ValueError as exc:
    print(f"Broadcasting Incompatibility Handled: {type(exc).__name__} -> {exc}")

Shape A: (4, 3, 1)
Shape B: (3, 5)
Broadcast Result Shape: (4, 3, 5)
Broadcasting Incompatibility Verified (3, 3)


### 51. How does `np.clip()` prevent gradient explosion and handle outlier bounds?

**Technical Breakdown**: `np.clip(a, a_min, a_max)` performs interval clamping. Given an input array, any value smaller than `a_min` is set to `a_min`, and any value larger than `a_max` is set to `a_max`. Values within the range $[\text{a\_min}, \text{a\_max}]$ remain unchanged. This operation is commonly used in neural networks to prevent exploding gradients and in tabular data preprocessing to winsorize extreme outliers. Time complexity is $\mathcal{O}(N)$ with $\mathcal{O}(N)$ memory allocation (or $\mathcal{O}(1)$ when used with the `out=arr` parameter).

In [51]:
import numpy as np

raw_gradients = np.array([-125.4, -2.3, 0.4, 5.1, 89.7, 450.2])
clipped_gradients = np.clip(raw_gradients, a_min=-5.0, a_max=5.0)

print("Original Values:", raw_gradients)
print("Clipped [-5.0, 5.0]:", clipped_gradients)

Original Values: [-1.254e+02 -2.300e+00  4.000e-01  5.100e+00  8.970e+01  4.502e+02]
Clipped [-5.0, 5.0]: [-5.  -2.3  0.4  5.   5.   5. ]


### 52. How does `np.unique()` compute unique values and frequency distributions?

**Technical Breakdown**: `np.unique(ar, return_counts=False, return_index=False, return_inverse=False)` flattens the input array (unless an axis is provided), sorts it, and returns sorted unique elements. Setting `return_counts=True` returns a second array containing occurrence counts for each unique element, providing histogram/frequency analysis in $\mathcal{O}(N \log N)$ time. Setting `return_inverse=True` returns an integer index array capable of reconstructing the original input array.

In [52]:
import numpy as np

labels = np.array(["cat", "dog", "dog", "bird", "cat", "dog"])

unique_vals, counts = np.unique(labels, return_counts=True)
print("Unique Items:", unique_vals)
print("Frequencies:", counts)

# Reconstruction with return_inverse
_, inv_indices = np.unique(labels, return_inverse=True)
print("Integer Category IDs:", inv_indices)

Unique Items: ['bird' 'cat' 'dog']
Frequencies: [1 2 3]
Integer Category IDs: [1 2 2 0 1 2]


### 53. How do `np.isnan()`, `np.nanmean()`, and `np.nan_to_num()` handle missing values?

**Technical Breakdown**: Standard statistical functions (`np.mean`, `np.sum`, `np.std`) propagate IEEE 754 `NaN` (Not a Number) values, returning `NaN` if any element in the input is missing. NumPy provides `nan`-prefix functions (e.g., `np.nanmean`, `np.nansum`) that compute reductions while ignoring `NaN` values. `np.isnan(arr)` returns a boolean mask marking `NaN` positions, and `np.nan_to_num(arr, nan=val)` replaces `NaN` values with user-specified fill values in-place or as a copy.

In [53]:
import numpy as np

measurements = np.array([10.0, 20.0, np.nan, 40.0, np.nan])

print("Standard np.mean:", np.mean(measurements))
print("NaN-Ignored np.nanmean:", np.nanmean(measurements))

# Detection and replacement
nan_mask = np.isnan(measurements)
cleaned_data = np.nan_to_num(measurements, nan=0.0)

print("NaN Mask:", nan_mask)
print("Imputed Array:", cleaned_data)


Standard np.mean: nan
NaN-Ignored np.nanmean: 23.333333333333332
NaN Mask: [False False  True False  True]
Imputed Array: [10. 20.  0. 40.  0.]
